# 基于Transformers的命名实体识别

## Step1 导入相关包

In [47]:
import evaluate
from datasets import load_dataset,load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer,DataCollatorForTokenClassification

## Step2 加载数据集

In [48]:
ner_datasets = load_from_disk("./ner_data/")
ner_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 20865
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 2319
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 4637
    })
})

In [49]:
ner_datasets["train"][0]

{'id': '0',
 'tokens': ['海',
  '钓',
  '比',
  '赛',
  '地',
  '点',
  '在',
  '厦',
  '门',
  '与',
  '金',
  '门',
  '之',
  '间',
  '的',
  '海',
  '域',
  '。'],
 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 5, 6, 0, 0, 0, 0, 0, 0]}

In [50]:
ner_datasets["train"].features

{'id': Value('string'),
 'tokens': List(Value('string')),
 'ner_tags': List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']))}

In [51]:
help(ner_datasets["train"].features["ner_tags"])

Help on List in module datasets.features.features object:

class List(Sequence)
 |  List(feature: Any, length: int = -1, id: Optional[str] = None) -> None
 |
 |  Feature type for large list data composed of child feature data type.
 |
 |  It is backed by `pyarrow.ListType`, which uses 32-bit offsets or a fixed length.
 |
 |  Args:
 |      feature ([`FeatureType`]):
 |          Child feature data type of each item within the large list.
 |      length (optional `int`, default to -1):
 |          Length of the list if it is fixed.
 |          Defaults to -1 which means an arbitrary length.
 |
 |  Method resolution order:
 |      List
 |      Sequence
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __eq__(self, other)
 |      Return self==value.
 |
 |  __init__(self, feature: Any, length: int = -1, id: Optional[str] = None) -> None
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  __replace__ = _replace(self, /, **changes) from dataclasses
 |
 |  __

In [52]:
label_list = ner_datasets["train"].features["ner_tags"].feature.names
label_list

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

## Step3 数据集预处理

In [53]:
tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base")

In [54]:
tokenizer(ner_datasets["train"][0]["tokens"], is_split_into_words=True) # 对于已经分词好的文本，需要设置is_split_into_words=True

{'input_ids': [101, 3862, 7157, 3683, 6612, 1765, 4157, 1762, 1336, 7305, 680, 7032, 7305, 722, 7313, 4638, 3862, 1818, 511, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [55]:
res = tokenizer("interesting word")
print(res)
# 由于存在subword tokenization，所以需要知道每个token对应的原始单词
dir(res)

{'input_ids': [101, 10673, 12865, 12921, 8181, 8681, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


['_MutableMapping__marker',
 '__abstractmethods__',
 '__annotate_func__',
 '__annotations_cache__',
 '__class__',
 '__class_getitem__',
 '__contains__',
 '__copy__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__ior__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__or__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__reversed__',
 '__ror__',
 '__setattr__',
 '__setitem__',
 '__setstate__',
 '__sizeof__',
 '__slots__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_encodings',
 '_n_sequences',
 'char_to_token',
 'char_to_word',
 'clear',
 'convert_to_tensors',
 'copy',
 'data',
 'encodings',
 'fromkeys',
 'get',
 'is_fast',
 'items',
 'keys',
 'n_seq

In [56]:
print(res.word_ids())
help(res.word_ids)

[None, 0, 0, 0, 0, 1, None]
Help on method word_ids in module transformers.tokenization_utils_base:

word_ids(batch_index: int = 0) -> list[int | None] method of transformers.tokenization_utils_base.BatchEncoding instance
    Return a list mapping the tokens to their actual word in the initial sentence for a fast tokenizer.

    Args:
        batch_index (`int`, *optional*, defaults to 0): The index to access in the batch.

    Returns:
        `list[Optional[int]]`: A list indicating the word corresponding to each token. Special tokens added by the
        tokenizer are mapped to `None` and other tokens are mapped to the index of their corresponding word
        (several tokens will be mapped to the same word index if they are parts of that word).



In [57]:
def process_function(batch): # 处理函数，输入是一个batch的数据，输出是一个tokenized的batch数据
    tokenized_batch = tokenizer(batch["tokens"], max_length=128, truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(batch["ner_tags"]):
        word_ids = tokenized_batch.word_ids(batch_index = i)
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            else:
                label_ids.append(label[word_id])
        labels.append(label_ids)
    tokenized_batch["labels"] = labels
    return tokenized_batch

In [58]:
tokenized_datasets = ner_datasets.map(process_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20865
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2319
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4637
    })
})

In [59]:
print(tokenized_datasets["train"][0])

{'id': '0', 'tokens': ['海', '钓', '比', '赛', '地', '点', '在', '厦', '门', '与', '金', '门', '之', '间', '的', '海', '域', '。'], 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 5, 6, 0, 0, 0, 0, 0, 0], 'input_ids': [101, 3862, 7157, 3683, 6612, 1765, 4157, 1762, 1336, 7305, 680, 7032, 7305, 722, 7313, 4638, 3862, 1818, 511, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 0, 0, 0, 0, 0, 0, 0, 5, 6, 0, 5, 6, 0, 0, 0, 0, 0, 0, -100]}


## Step4 创建模型

In [60]:
model = AutoModelForTokenClassification.from_pretrained("hfl/chinese-macbert-base", num_labels=len(label_list))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: hfl/chinese-macbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be i

In [61]:
configuration = model.config
configuration

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "directionality": "bidi",
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5,
    "LABEL_6": 6
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_la

In [62]:
configuration.num_labels

7

In [63]:
dir(model.config)

['__annotate_func__',
 '__annotations_cache__',
 '__class__',
 '__class_validators__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__dataclass_transform__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__post_init__',
 '__reduce__',
 '__reduce_ex__',
 '__replace__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slotnames__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__validators__',
 '__weakref__',
 '_attn_implementation',
 '_attn_implementation_internal',
 '_auto_class',
 '_check_received_keys',
 '_commit_hash',
 '_decode_special_floats',
 '_dict_from_json_file',
 '_encode_special_floats',
 '_experts_implementation',
 '_experts_implementation_internal',
 '_get_config_dict',
 '_get_files_timestamps',
 '_ge

In [64]:
from transformers import BertConfig
help(BertConfig)

Help on class BertConfig in module transformers.models.bert.configuration_bert:

class BertConfig(transformers.configuration_utils.PreTrainedConfig)
 |  BertConfig(
 |      transformers_version: str | None = None,
 |      architectures: list[str] | None = None,
 |      output_hidden_states: bool | None = False,
 |      return_dict: bool | None = True,
 |      dtype: Union[str, 'torch.dtype'] | None = None,
 |      chunk_size_feed_forward: int = 0,
 |      is_encoder_decoder: bool = False,
 |      id2label: dict[int, str] | dict[str, str] | None = None,
 |      label2id: dict[str, int] | dict[str, str] | None = None,
 |      problem_type: Literal['regression', 'single_label_classification', 'multi_label_classification'] | None = None,
 |      *,
 |      vocab_size: int = 30522,
 |      hidden_size: int = 768,
 |      num_hidden_layers: int = 12,
 |      num_attention_heads: int = 12,
 |      intermediate_size: int = 3072,
 |      hidden_act: str = 'gelu',
 |      hidden_dropout_prob: fl

In [65]:
# configuration.num_labels = len(label_list)
# model = AutoModelForTokenClassification.from_pretrained("hfl/chinese-macbert-base", config = configuration)

# 也可以直接在加载模型的时候指定num_labels参数，模型会自动根据这个参数修改config中的num_labels参数
# model = AutoModelForTokenClassification.from_pretrained(
#     "hfl/chinese-macbert-base", num_labels = len(label_list)
# )
# model.config.num_labels

## Step5 创建评估函数

In [66]:
seqeval = evaluate.load("seqeval")
seqeval

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "d:\code_local\Python\transformers-code\.venv\Lib\site-packages\huggingface_hub\utils\_http.py", line 761, in hf_raise_for_status
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "d:\code_local\Python\transformers-code\.venv\Lib\site-packages\httpx\_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '403 Forbidden' for url 'https://huggingface.co/api/models/hfl/chinese-macbert-base/discussions?p=0'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\YYH\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File 

EvaluationModule(name: "seqeval", module_type: "metric", features: {'predictions': List(Value('string')), 'references': List(Value('string'))}, usage: """
Produces labelling scores along with its sufficient statistics
from a source against one or more references.

Args:
    predictions: List of List of predicted labels (Estimated targets as returned by a tagger)
    references: List of List of reference labels (Ground truth (correct) target values)
    suffix: True if the IOB prefix is after type, False otherwise. default: False
    scheme: Specify target tagging scheme. Should be one of ["IOB1", "IOB2", "IOE1", "IOE2", "IOBES", "BILOU"].
        default: None
    mode: Whether to count correct entity labels with incorrect I/B tags as true positives or not.
        If you want to only count exact matches, pass mode="strict". default: None.
    sample_weight: Array-like of shape (n_samples,), weights for individual samples. default: None
    zero_division: Which value to substitute as a

In [67]:
import numpy as np
def eval_metric(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis= -1)

    true_predictions = [
        [p for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [l for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(
        predictions=true_predictions, references=true_labels, mode="strict"
    )
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    "accuracy": results["overall_accuracy"],
    }
    

## Step6 配置训练参数

In [68]:
args = TrainingArguments(
    output_dir="models_for_ner",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="f1",
    load_best_model_at_end=True,
    logging_steps=50
)

## Step7 创建Trainer

In [69]:
# 使用少量数据调试
train_dataset = tokenized_datasets["train"].select(range(len(tokenized_datasets["train"]) // 10))
eval_dataset = tokenized_datasets["validation"].select(range(len(tokenized_datasets["validation"]) // 10))

In [70]:
trainer = Trainer(
    model= model,
    args= args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics= eval_metric,
    data_collator= DataCollatorForTokenClassification(tokenizer)
)

## Step8 训练模型

In [71]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 